# Hapi

## Conceptual Distributed Hydrological Model

- Please change the Path in the following cell to the directory where you stored the case study data

In [ ]:
Comp = "F:/01Algorithms/Hydrology/HAPI"
import os

os.chdir("../../../")
os.getcwd()

### Import Modules

In [ ]:
import datetime as dt

import Hapi.rrm.hbv_bergestrom92 as HBV
from Hapi.catchment import Catchment
from Hapi.run import Run

### Paths

In [ ]:
Path = "Examples/Hydrological model/data/distributed_model"
PrecPath = Path + "/prec"
Evap_Path = Path + "/evap"
TempPath = Path + "/temp"
FlowAccPath = Path + "/GIS/acc4000.tif"
FlowDPath = Path + "/GIS/fd4000.tif"

ParPathRun = Path + "/Parameter set-Avg/"

### Meteorological data

In [ ]:
AreaCoeff = 1530
InitialCond = [0, 5, 5, 5, 0]
Snow = 0
"""
Create the model object and read the input data
"""
start = "2009-01-01"
end = "2009-04-10"
name = "Coello"
Coello = Catchment(name, start, end, SpatialResolution="Distributed")
Coello.readRainfall(PrecPath)
Coello.readTemperature(TempPath)
Coello.readET(Evap_Path)

Coello.readFlowAcc(FlowAccPath)
Coello.readFlowDir(FlowDPath)
Coello.readParameters(ParPathRun, Snow)
Coello.readLumpedModel(HBV, AreaCoeff, InitialCond)

## Gauges

In [ ]:
Coello.readGaugeTable(Path + "/stations/gauges.csv", FlowAccPath)
GaugesPath = Path + "/stations/"
Coello.readDischargeGauges(GaugesPath, column='id', fmt="%Y-%m-%d")

In [ ]:
Coello.GaugesTable

# Run the model

Outputs:
----------
    1-statevariables: [numpy attribute]
        4D array (rows,cols,time,states) states are [sp,wc,sm,uz,lv]
    2-qlz: [numpy attribute]
        3D array of the lower zone discharge
    3-quz: [numpy attribute]
        3D array of the upper zone discharge
    4-qout: [numpy attribute]
        1D timeseries of discharge at the outlet of the catchment
        of unit m3/sec
    5-quz_routed: [numpy attribute]
        3D array of the upper zone discharge  accumulated and
        routed at each time step
    6-qlz_translated: [numpy attribute]
        3D array of the lower zone discharge translated at each time step


In [ ]:
Run.run_distributed(Coello)

In [ ]:
import numpy as np

np.shape(Coello.results.q_total)

In [ ]:
Coello.GaugesTable['area ratio'].tolist()

In [ ]:
Coello.extractDischarge()

for i in range(len(Coello.GaugesTable)):
    gaugeid = Coello.GaugesTable.loc[i, 'id']
    print("----------------------------------")
    print("Gauge - " + str(gaugeid))
    print("RMSE= " + str(round(Coello.Metrics.loc['RMSE', gaugeid], 2)))
    print("NSE= " + str(round(Coello.Metrics.loc['NSE', gaugeid], 2)))
    print("NSEhf= " + str(round(Coello.Metrics.loc['NSEhf', gaugeid], 2)))
    print("KGE= " + str(round(Coello.Metrics.loc['KGE', gaugeid], 2)))
    print("WB= " + str(round(Coello.Metrics.loc['WB', gaugeid], 2)))
    print("Pearson CC= " + str(round(Coello.Metrics.loc['Pearson-CC', gaugeid], 2)))
    print("R2 = " + str(round(Coello.Metrics.loc['R2', gaugeid], 2)))

In [ ]:
Coello.Metrics

In [ ]:
Coello.results.q_total[0, 0, 0]

### Calculate performance criteria

### Plot Hydrographs

In [ ]:
gaugei = 5
plotstart = "2009-01-01"
plotend = "2009-04-10"

Coello.plotHydrograph(plotstart, plotend, gaugei)

In [ ]:
Coello.ListAttributes()

In [ ]:
import matplotlib.pyplot as plt

plt.plot(Coello.results.q_total[12, 1, :])

### Animation

In [ ]:
% matplotlib inline
from IPython.display import HTML

plotstart = "2009-01-01"
plotend = "2009-01-20"
Option = 5
threshold = 10

Anim = Coello.plotDistributedResults(
    plotstart,
    plotend,
    Figsize=(9, 9),
    Option=Option,
    threshold=160,
    PlotNumbers=False,
    TicksSpacing=5,
    Interval=200,
    Gauges=True,
    cmap='inferno',
    Textloc=[0.1, 0.2],
    Gaugecolor='red',
    ColorScale=2,
    IDcolor='blue',
    IDsize=25,
)
HTML(Anim.to_html5_video())

- if you like to save the animation

In [ ]:
SaveTo = Path + "anim.mov"
Coello.saveAnimation(VideoFormat="mov", Path=SaveTo, SaveFrames=3)

# 11-Store the result into rasters

In [ ]:
StartDate = "2009-01-01"
EndDate = "2010-04-20"
Prefix = 'Qtot_'
SaveTo = Path + "/results/"
Coello.saveResults(
    FlowAccPath,
    Result=1,
    StartDate=StartDate,
    EndDate=EndDate,
    Path=SaveTo,
    Prefix=Prefix,
)

In [ ]:
Path